# Simulated Annealing (Numba, Parallel)

In [98]:
import numpy as np
import math
import time
import ast
import random as rd
import matplotlib.pyplot as plt
from numba import njit, prange

In [99]:
# Parse data code
def parse_tsplib(file_path):
    """Parses standard EUC_2D coordinate sections from a .tsp file."""
    clean_lines = (line for line in open(file_path) if len(line.split()) == 3 and line.split()[0].isdigit())
    coords = np.loadtxt(clean_lines, usecols=(1, 2))
    return coords

def get_data(file_path):
    tsp_data = []
    with open(file_path, "r") as f:
        next(f)
        for line in f:
            parts = line.split()
            if parts[2] == "EUC_2D":
                name, nb_city, data_type = parts[0], int(parts[1]), parts[2]
                bound = ast.literal_eval(parts[3])
                tsp_data.append((name, nb_city, data_type, bound))
    return tsp_data


In [100]:
# Simulated Annealing Code
@njit(fastmath=True)
def dist(u, v, coords):
    """Calculates the Euclidean distance between two city indices."""
    x1, y1 = coords[u, 0], coords[u, 1]
    x2, y2 = coords[v, 0], coords[v, 1]
    return np.sqrt((x1 - x2)**2 + (y1 - y2)**2)


@njit
def two_opt_inplace(circuit, a, b):
    """Reverses the segment of circuit from index a to b-1 in-place."""
    i = a
    j = b - 1
    while i < j:
        temp = circuit[i]
        circuit[i] = circuit[j]
        circuit[j] = temp
        i += 1
        j -= 1

@njit
def nearest_neighbor_numba(coords, start_node):
    """Generates a high-quality greedy starting tour starting from start_node."""
    n = len(coords)
    visited = np.zeros(n, dtype=np.bool_)
    tour = np.empty(n + 1, dtype=np.int32)
    
    current = start_node
    tour[0] = current
    visited[current] = True
    
    for i in range(1, n):
        min_d = np.inf
        nearest = -1
        x1, y1 = coords[current, 0], coords[current, 1]
        
        for j in range(n):
            if not visited[j]:
                x2, y2 = coords[j, 0], coords[j, 1]
                d = np.sqrt((x1 - x2)**2 + (y1 - y2)**2)
                if d < min_d:
                    min_d = d
                    nearest = j
                    
        tour[i] = nearest
        visited[nearest] = True
        current = nearest
        
    tour[n] = start_node
    return tour

@njit
def simulated_annealing_numba(coords, init_temp, cooling_rate, mini_temp, loop_per_temp, initial_tour):
    n = len(coords)
    
    # 1. Initialize circuit as a NumPy array of 32-bit integers
    circuit = initial_tour.copy()
    
    # 2. Calculate initial cost
    cost = 0.0
    for i in range(n):
        cost += dist(circuit[i], circuit[i+1], coords)
        
    temp = init_temp
    bestcircuit = np.empty_like(circuit)
    bestcircuit[:] = circuit
    bestcost = cost

    while temp > mini_temp:
        for _ in range(loop_per_temp):
            # np.random.randint(low, high) is half-open, so we use n + 1 to include n
            a = np.random.randint(1, n + 1)
            b = np.random.randint(1, n + 1)
            
            if a == b:
                continue
                
            # Fast manual sorting so a < b
            if a > b:
                temp_idx = a
                a = b
                b = temp_idx
                
            u1, u2 = circuit[a - 1], circuit[a]
            v1, v2 = circuit[b - 1], circuit[b]
            
            # O(1) delta cost calculations
            len_before = dist(u1, u2, coords) + dist(v1, v2, coords)
            len_after = dist(u1, v1, coords) + dist(u2, v2, coords)
            dE = len_after - len_before
            
            # Acceptance test
            if dE < 0 or np.random.random() < np.exp(-dE / temp):
                two_opt_inplace(circuit, a, b)
                cost += dE

                if cost < bestcost:
                    bestcost = cost
                    bestcircuit[:] = circuit
                
        temp *= cooling_rate
        
    return bestcircuit, bestcost

@njit(parallel=True)
def run_multiple_sa_parallel(coords, init_temp, cooling_rate, mini_temp, loop_per_temp, nb_runs):
    result_array = np.empty(nb_runs, dtype=np.float64)
    n = len(coords)
    
    # prange tells Numba to execute these loop iterations in parallel across CPU cores
    for i in prange(nb_runs):
        start_node = np.random.randint(0, n)
        initial_tour = nearest_neighbor_numba(coords, start_node)
        _, bound = simulated_annealing_numba(coords, init_temp, cooling_rate, mini_temp, loop_per_temp, initial_tour)
        result_array[i] = bound
        
    return result_array

In [101]:
# SOM code
@njit(fastmath=True)
def solve_tsp_som(coords, epochs=150, lr_init=0.8):
    """
    Highly optimized SOM TSP solver with reduced neuron density (1.3N)
    and adjustable epochs to match Simulated Annealing performance.
    """
    n = len(coords)
    m = int(1.3 * n)  # Optimized neuron density (1.3x instead of 2.0x)
    
    xs = coords[:, 0]
    ys = coords[:, 1]
    center_x = np.mean(xs)
    center_y = np.mean(ys)
    
    neurons = np.empty((m, 2))
    r = 0.05 * (np.max(xs) - np.min(xs))
    for i in range(m):
        angle = 2.0 * np.pi * i / m
        neurons[i, 0] = center_x + r * np.cos(angle)
        neurons[i, 1] = center_y + r * np.sin(angle)
        
    sigma_init = m / 10.0
    
    for epoch in range(epochs):
        pct = epoch / epochs
        lr = lr_init * (1.0 - pct)
        sigma = sigma_init * (1.0 - pct)
        if sigma < 1.0:
            sigma = 1.0
            
        limit = int(3.0 * sigma) + 1
        h_table = np.empty(limit + 1)
        two_sigma_sq = 2.0 * sigma * sigma
        for offset in range(limit + 1):
            h_table[offset] = np.exp(-(offset * offset) / two_sigma_sq)
            
        shuffled_indices = np.random.permutation(n)
        
        for city_idx in shuffled_indices:
            city_x = coords[city_idx, 0]
            city_y = coords[city_idx, 1]
            
            # Find BMU (1.3x faster due to smaller m)
            bmu_idx = 0
            min_dist_sq = np.inf
            for j in range(m):
                dx = neurons[j, 0] - city_x
                dy = neurons[j, 1] - city_y
                d_sq = dx*dx + dy*dy
                if d_sq < min_dist_sq:
                    min_dist_sq = d_sq
                    bmu_idx = j
            
            # Update BMU and neighbors
            for offset in range(-limit, limit + 1):
                j = (bmu_idx + offset) % m
                h = h_table[np.abs(offset)]
                neurons[j, 0] += lr * h * (city_x - neurons[j, 0])
                neurons[j, 1] += lr * h * (city_y - neurons[j, 1])
                
    # Reconstruct Tour
    city_to_neuron = np.empty(n, dtype=np.int32)
    for i in range(n):
        city_x = coords[i, 0]
        city_y = coords[i, 1]
        bmu_idx = 0
        min_dist_sq = np.inf
        for j in range(m):
            dx = neurons[j, 0] - city_x
            dy = neurons[j, 1] - city_y
            d_sq = dx*dx + dy*dy
            if d_sq < min_dist_sq:
                min_dist_sq = d_sq
                bmu_idx = j
        city_to_neuron[i] = bmu_idx
        
    sorted_indices = np.argsort(city_to_neuron)
    
    circuit = np.empty(n + 1, dtype=np.int32)
    for i in range(n):
        circuit[i] = sorted_indices[i]
    circuit[n] = sorted_indices[0]
    
    return circuit

@njit(fastmath=True)
def calc_dist(coords, circuit):
    """
    Calculates the total Euclidean length of a given closed TSP circuit.
    Optimized via Numba JIT compilation.
    """
    cost = 0.0
    n = len(coords)  # Number of cities
    for i in range(n):
        u = circuit[i]
        v = circuit[i+1]
        
        # Inlined Euclidean distance calculation
        dx = coords[u, 0] - coords[v, 0]
        dy = coords[u, 1] - coords[v, 1]
        cost += np.sqrt(dx*dx + dy*dy)
        
    return cost

In [102]:
# List data
tsp_data = get_data("tsp_data.txt")
print(tsp_data)

[('a280', 280, 'EUC_2D', 2579), ('berlin52', 52, 'EUC_2D', 7542), ('bier127', 127, 'EUC_2D', 118282), ('brd14051', 14051, 'EUC_2D', [468942, 469445]), ('ch130', 130, 'EUC_2D', 6110), ('ch150', 150, 'EUC_2D', 6528), ('d198', 198, 'EUC_2D', 15780), ('d493', 493, 'EUC_2D', 35002), ('d657', 657, 'EUC_2D', 48912), ('d1291', 1291, 'EUC_2D', 50801), ('d1655', 1655, 'EUC_2D', 62128), ('d2103', 2103, 'EUC_2D', [79952, 80450]), ('d15112', 15112, 'EUC_2D', [1564590, 1573152]), ('d18512', 18512, 'EUC_2D', [644650, 645488]), ('eil51', 51, 'EUC_2D', 426), ('eil76', 76, 'EUC_2D', 538), ('eil101', 101, 'EUC_2D', 629), ('fl417', 417, 'EUC_2D', 11861), ('fl1400', 1400, 'EUC_2D', 20127), ('fl1577', 1577, 'EUC_2D', [22204, 22249]), ('fl3795', 3795, 'EUC_2D', [28723, 28772]), ('fnl4461', 4461, 'EUC_2D', 182566), ('gil262', 262, 'EUC_2D', 2378), ('kroA100', 100, 'EUC_2D', 21282), ('kroB100', 100, 'EUC_2D', 22141), ('kroC100', 100, 'EUC_2D', 20749), ('kroD100', 100, 'EUC_2D', 21294), ('kroE100', 100, 'EUC_2D

In [124]:
# Select data
current_tsp = next((r for r in tsp_data if r[0] == "d15112"))
cities = parse_tsplib(f"tsp_data/{current_tsp[0]}.tsp")
print("Current data:", current_tsp[0])
print(cities[:5])
print("Bound:", current_tsp[3])

Current data: d15112
[[ 5826.  1350.]
 [  413. 10751.]
 [ 8419.  4442.]
 [ 4317.  8968.]
 [ 7196. 20891.]]
Bound: [1564590, 1573152]


In [125]:
# Run parallel SA
NB_RUN = 4; INIT_TEMP = 10.0; COOLING = 0.99; MINI_TEMP = 0.01; LOOP_PER_TEMP = len(cities)

start = time.perf_counter()
result_array = run_multiple_sa_parallel(cities, INIT_TEMP, COOLING, MINI_TEMP, LOOP_PER_TEMP, NB_RUN)
end = time.perf_counter()

avg_length = np.mean(result_array)
min_length = np.min(result_array)
print(f"Execution time for {current_tsp[0]}, N={current_tsp[1]}: {(end-start):0.4f}")
print(f"Average length: {int(avg_length)}")
print(f"Minimum length: {int(min_length)}")
print("True bound:", current_tsp[3])
print("Error: ", 100*np.abs(min_length-current_tsp[3])/current_tsp[3], "%")

Execution time for d15112, N=15112: 1.9566
Average length: 1898757
Minimum length: 1893835
True bound: [1564590, 1573152]
Error:  [21.04355185 20.38476307] %


In [120]:
# Run Self-Organizing Map
start = time.perf_counter()
som_circuit = solve_tsp_som(cities, epochs=150)
som_cost = calc_dist(cities, som_circuit)
end = time.perf_counter()

print(f"Execution time for {current_tsp[0]}, N={current_tsp[1]}: {(end-start):0.4f}")
print(f"Minimum length: {int(som_cost)}")
print("True bound:", current_tsp[3])
print("Error: ", 100*np.abs(som_cost-current_tsp[3])/current_tsp[3], "%")

Execution time for d1655, N=1655: 0.8249
Minimum length: 102626
True bound: 62128
Error:  65.18490748580145 %
